# Track A 07: Qwen2.5-3B Adapter and Parity Pilot

This self-contained Colab notebook validates the Qwen-specific adapter before any Qwen energy or full quality matrix is built. It writes only to `CSE407_qwen_track_a_adapter_pilot_v1/`.

| Label | Pilot path |
|---|---|
| Q0 | FP16 architecture gate only |
| Q1-S | Stock AWQ parity control |
| Q1-W | Matched-wrapper AWQ baseline |
| Q2 | AWQ + custom pyramidal eviction |
| Q3-Ref | AWQ + `turboquant==0.2.0` numerical reference path |
| Q4-Ref | Sequential eviction then TQ-Ref |

TQ-Ref is not packed, fused, or physically 4-bit. This pilot creates no energy rows and does not run the 300-row quality matrix.


## Run Order and Fail-Closed Boundary

Run cells in numeric order on a Colab T4. The notebook verifies immutable Hub revisions, the exact Transformers 5.12.1 Qwen2 source, tokenizer hashes, frozen semantics, and the new Qwen prompt hash before loading a model. Q0 is loaded and released before the AWQ phase. The final validation JSON is written only after every architecture, parity, position, cache, eviction, TQ-Ref, active-use, and storage gate passes.

This source notebook is intentionally unexecuted. Do not widen tolerances, change hashes, reuse a result directory, or continue to the full Qwen matrix after a failed gate.


## 1. Install the Pinned Reference Stack

Run this cell once, choose **Runtime > Restart session**, then continue with section 2. The Transformers 5.12.1 AWQ implementation uses GPTQModel; AutoAWQ is intentionally not installed because it can downgrade Transformers.


In [ ]:
%pip install -q "transformers==5.12.1" "gptqmodel==7.1.0" "turboquant==0.2.0"
%pip install -q "accelerate==1.13.0" "safetensors==0.7.0" "tokenizers==0.22.2" "huggingface-hub==1.5.0"
%pip install -q "numpy==2.2.6" "scipy==1.16.1" ninja maturin


## 2. Runtime, Immutable Revisions, and Source Gate

The Qwen2 API in Transformers 5.12.1 has no `cache_position` argument. This notebook therefore requires explicit absolute `position_ids` during continuation and bypasses the shared mask factory only for the single-token, unpadded decode probe. The wrapper still forwards the exact source mask object during the full 4096-token prefill.


In [ ]:
import gc
import hashlib
import importlib.metadata
import inspect
import json
import math
import os
import random
import re
import time
from collections.abc import Mapping
from pathlib import Path

import numpy as np
import scipy
import torch
import torch.nn.functional as F
import transformers
import transformers.cache_utils as transformers_cache_utils
import transformers.masking_utils as transformers_masking_utils
import transformers.integrations.sdpa_attention as transformers_sdpa_attention
import turboquant
from huggingface_hub import HfApi
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, DynamicCache, set_seed
from transformers.modeling_utils import ALL_ATTENTION_FUNCTIONS
from transformers.models.qwen2.modeling_qwen2 import (
    Qwen2Attention,
    apply_rotary_pos_emb,
    eager_attention_forward,
    repeat_kv,
)
from turboquant import TurboQuantCache, TurboQuantMSE

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

SEED = 407
SCHEMA_VERSION = "track-a-qwen-adapter-pilot-v1-2026-08-04-1"
EXPECTED_FINGERPRINT = "91def77c920729e9"
MODEL_FP16 = "Qwen/Qwen2.5-3B-Instruct"
MODEL_FP16_REVISION = "aa8e72537993ba99e69dfaafa59ed015b17504d1"
MODEL_AWQ = "Qwen/Qwen2.5-3B-Instruct-AWQ"
MODEL_AWQ_REVISION = "3559b226e8ce77211e2c1bd7ddfb7686fec4d6dd"
TRANSFORMERS_COMMIT = "ddb849abe009d1089e6c691bfc897f27211c663c"
EXPECTED_QWEN2_SOURCE_SHA256 = "99fa98c5676604cf6ef505892b70fda5c1c4cd835971459f38d61090cccab1e4"
EXPECTED_CACHE_UTILS_SOURCE_SHA256 = "2783b2702910dbb6e70aa05a093f71e1c51c110d7fa73c904fc38d12e5882545"
EXPECTED_MASKING_UTILS_SOURCE_SHA256 = "aa525d800f824b25cbe9542ca7cc5b35e948e7cfa3178e2c0f2356a15ff32df2"
EXPECTED_SDPA_SOURCE_SHA256 = "87f933d1a2d8508df572da5c0748c6b24c22ff2b625796949957dcd86cc57564"
EXPECTED_TQ_SOURCE_SHA256 = "7358e2778c8de6974b0c7212ae8e3dbe00552a96b2e2df0aa42878f239a4c72a"
EXPECTED_SEMANTIC_SHA256 = "fffa82607af6e7d2c892d18d5bc790524570833472ad397b7ab2e54eb03005ef"
EXPECTED_PROMPT_SET_SHA256 = "115c1568fea7aeea038a46ce309d0c8bcaf1f5405544466ad6f379ee7e94ae3a"
EXPECTED_TOKENIZER_BACKEND_SHA256 = "e360e6f6266884e6667874f3a7d158ca20c4639e2e1dc80427e1edc2c69a38c7"
EXPECTED_CHAT_TEMPLATE_SHA256 = "cd8e9439f0570856fd70470bf8889ebd8b5d1107207f67a5efb46e342330527f"
EXPECTED_SPECIAL_TOKENS_SHA256 = "edcb2fc2acbbe77f858a9c4fe51295ffdb84711efba5703ec5906b3d67282569"

CONTEXT_TOKENS = 4096
MAX_NEW_TOKENS = 16
DEFAULT_AVERAGE_BUDGET = 1024
DEFAULT_TQ_BITS = 4
TQ_NMSE_MAX = 0.25
NUM_LAYERS = 36
HIDDEN_SIZE = 2048
INTERMEDIATE_SIZE = 11008
NUM_Q_HEADS = 16
NUM_KV_HEADS = 2
HEAD_DIM = 128
VOCAB_SIZE = 151936
MAX_POSITIONS = 32768
ROPE_THETA = 1_000_000.0
WINDOW = 32
POOL_KERNEL = 5
PYRAMID_BETA = 20
PARITY_ATOL = 5e-3
PARITY_RTOL = 5e-3
PARITY_SCENARIOS = ("d10-s00", "d50-s00", "d90-s00")
PROBE_SCENARIO = "d50-s00"
VARIANTS = ("Q0", "Q1-S", "Q1-W", "Q2", "Q3-Ref", "Q4-Ref")
PILOT_ARTIFACT_NAME = "CSE407_qwen_track_a_adapter_pilot_v1"
FORBIDDEN_ARTIFACT_NAMES = {
    "CSE407_llama_v5_final",
    "CSE407_llama_track_a_full_v1",
    "CSE407_llama_power_robustness_v1",
    "CSE407_llama_track_a_pilot_discarded",
    "CSE407_llama_track_a_pilot_v2_discarded",
    "CSE407_llama_track_a_pilot_v3_discarded",
    "CSE407_llama_track_a_pilot_v4_discarded",
    "CSE407_qwen_track_a_reference_v1",
}

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), "A CUDA GPU is required"
GPU = torch.cuda.get_device_properties(0)
assert "T4" in GPU.name, f"Expected an NVIDIA T4, found {GPU.name!r}"
assert transformers.__version__ == "5.12.1"
assert importlib.metadata.version("gptqmodel") == "7.1.0"
assert importlib.metadata.version("turboquant") == "0.2.0"
assert importlib.metadata.version("tokenizers") == "0.22.2"
assert np.__version__ == "2.2.6"
assert scipy.__version__ == "1.16.1"

qwen2_source_path = Path(inspect.getsourcefile(Qwen2Attention))
qwen2_source_sha256 = hashlib.sha256(qwen2_source_path.read_bytes()).hexdigest()
assert qwen2_source_sha256 == EXPECTED_QWEN2_SOURCE_SHA256
forward_parameters = tuple(inspect.signature(Qwen2Attention.forward).parameters)
assert forward_parameters == (
    "self", "hidden_states", "position_embeddings", "attention_mask", "past_key_values", "kwargs"
), forward_parameters
assert "cache_position" not in forward_parameters

dependent_source_hashes = {
    "cache_utils": hashlib.sha256(Path(inspect.getsourcefile(transformers_cache_utils)).read_bytes()).hexdigest(),
    "masking_utils": hashlib.sha256(Path(inspect.getsourcefile(transformers_masking_utils)).read_bytes()).hexdigest(),
    "sdpa_attention": hashlib.sha256(Path(inspect.getsourcefile(transformers_sdpa_attention)).read_bytes()).hexdigest(),
}
assert dependent_source_hashes == {
    "cache_utils": EXPECTED_CACHE_UTILS_SOURCE_SHA256,
    "masking_utils": EXPECTED_MASKING_UTILS_SOURCE_SHA256,
    "sdpa_attention": EXPECTED_SDPA_SOURCE_SHA256,
}

tq_package_dir = Path(turboquant.__file__).parent
tq_source_sha256 = hashlib.sha256(
    b"".join(path.read_bytes() for path in sorted(tq_package_dir.glob("*.py")))
).hexdigest()
assert tq_source_sha256 == EXPECTED_TQ_SOURCE_SHA256

api = HfApi()
assert api.model_info(MODEL_FP16, revision=MODEL_FP16_REVISION).sha == MODEL_FP16_REVISION
assert api.model_info(MODEL_AWQ, revision=MODEL_AWQ_REVISION).sha == MODEL_AWQ_REVISION

from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = Path("/content/drive/MyDrive") / PILOT_ARTIFACT_NAME
assert SAVE_DIR.name == PILOT_ARTIFACT_NAME
assert SAVE_DIR.name not in FORBIDDEN_ARTIFACT_NAMES
SEMANTIC_MANIFEST_JSON = SAVE_DIR / "frozen_semantic_manifest.json"
PROMPT_MANIFEST_JSON = SAVE_DIR / "frozen_qwen_prompt_manifest.json"
ENVIRONMENT_JSON = SAVE_DIR / "qwen_adapter_environment.json"
PARITY_JSON = SAVE_DIR / "qwen_q1s_q1w_parity.json"
VALIDATION_JSON = SAVE_DIR / "qwen_adapter_validation.json"
SESSION_ID = time.strftime("%Y%m%dT%H%M%S") + "-" + os.urandom(4).hex()
print(GPU.name, "source and immutable-revision gates passed")


## 3. Embedded Frozen V4 Semantic Generator

This is the accepted Notebook 05 generator, including the historical draft status string that participates in the frozen semantic hash. It is embedded so the pilot does not depend on another notebook at execution time. Scenario facts and ordering must not be changed for Qwen.


In [ ]:
GENERATOR_VERSION = "track-a-hard-retrieval-v4-controlled-pairs-draft"
MANIFEST_SCHEMA = "cse407-hard-retrieval-manifest-v1"
BASE_SEED = 407_100
DEPTHS = (0.1, 0.5, 0.9)
SCENARIOS_PER_DEPTH = 20
RECORD_COUNT = 32
FINAL_QUERY_WINDOW = 32
MIN_RECORD_GAP_TOKENS = 16

CODE_PATTERN = re.compile(
    r"(?<![A-Z0-9])([A-H]\d{3}-[K-NP-Q]\d{2})(?![A-Z0-9])",
    re.IGNORECASE,
)
PARTIAL_CODE_PATTERN = re.compile(
    r"(?<![A-Z0-9])(?:[A-H]\d{1,2}|[A-H]\d{3}(?:-(?:[K-NP-Q]?\d?)?)?)(?![A-Z0-9])",
    re.IGNORECASE,
)
SYSTEM_INSTRUCTION = (
    "Use only the document. Return exactly one activation code and no explanation."
)
PLACEHOLDER = "{HARD_RETRIEVAL_DOCUMENT}"
KEY_LABELS = (
    "AURORA", "BOREAL", "CEDAR", "DRAGON", "EMBER", "FALCON", "GARNET", "HARBOR",
    "INDIGO", "JASPER", "KESTREL", "LANTERN", "MARINER", "NEBULA", "ORBIT", "PRAIRIE",
    "QUARTZ", "RAVEN", "SUMMIT", "TEMPEST", "UMBRA", "VECTOR", "WILLOW", "XENON",
    "YARROW", "ZEPHYR", "ORBITAL", "MARINA", "CEDRIC", "FALCONE", "GARNET-X", "HARBOR-X",
)
FILLER_TOPICS = (
    "thermal scheduling", "memory allocation", "coastal monitoring",
    "warehouse routing", "battery maintenance", "traffic forecasting",
    "compiler testing", "water treatment", "satellite imaging",
    "clinical logistics", "network telemetry", "acoustic sensing",
    "grid balancing", "crop analysis", "robot calibration", "archive indexing",
)
FILLER_METHODS = (
    "compares recent measurements with a rolling baseline",
    "records each decision before the next review cycle",
    "uses independent checks to detect inconsistent observations",
    "separates routine maintenance from exceptional interventions",
    "prioritizes stable operation over short-lived peak performance",
    "tracks uncertainty alongside every summarized measurement",
    "repeats validation after any material configuration change",
    "keeps raw observations available for later auditing",
)
FILLER_OUTCOMES = (
    "The review found that seasonal variation explained most of the change.",
    "Operators retained the conservative threshold for the next evaluation.",
    "No single measurement was treated as sufficient evidence for deployment.",
    "The team documented both successful trials and inconclusive observations.",
    "A later audit confirmed the ordering but revised the uncertainty estimate.",
    "The report emphasized reproducibility and clearly separated derived values.",
    "Follow-up work focused on failure recovery rather than headline throughput.",
    "The final recommendation required another independent measurement period.",
)


def canonical_hash(value):
    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def token_ids_hash(input_ids):
    return hashlib.sha256(",".join(map(str, input_ids)).encode("ascii")).hexdigest()


def _make_unique_codes(rng, count):
    letters = "ABCDEFGH"
    suffixes = "KLMNPQ"
    codes = set()
    while len(codes) < count:
        codes.add(
            f"{rng.choice(letters)}{rng.randrange(100, 1000):03d}-"
            f"{rng.choice(suffixes)}{rng.randrange(10, 100):02d}"
        )
    result = sorted(codes)
    rng.shuffle(result)
    return result


def _make_similar_keys(rng, count):
    candidates = [f"NODE-{label}" for label in KEY_LABELS]
    if count > len(candidates):
        raise ValueError("Controlled-pair pilot supports at most 32 records")
    rng.shuffle(candidates)
    return candidates[:count]


def validate_semantic_scenario(scenario):
    records = scenario["records"]
    if not 32 <= len(records) <= 64:
        raise ValueError("Each scenario must contain 32 to 64 records")
    keys = [record["key"] for record in records]
    codes = [record["code"] for record in records]
    if len(set(keys)) != len(keys) or len(set(codes)) != len(codes):
        raise ValueError("Record keys and codes must be unique")
    if any(CODE_PATTERN.fullmatch(code) is None for code in codes):
        raise ValueError("Every record code must match the declared answer format")
    targets = [record for record in records if record["is_target"]]
    if len(targets) != 1:
        raise ValueError("Each scenario must have exactly one target record")
    target = targets[0]
    if target["key"] != scenario["target_key"] or target["code"] != scenario["target_code"]:
        raise ValueError("Target metadata does not match the target record")
    if scenario["depth"] not in DEPTHS:
        raise ValueError(f"Unsupported target depth: {scenario['depth']}")


def build_semantic_manifest(
    depths=DEPTHS,
    scenarios_per_depth=SCENARIOS_PER_DEPTH,
    record_count=RECORD_COUNT,
    base_seed=BASE_SEED,
):
    if not 32 <= record_count <= 64:
        raise ValueError("record_count must be between 32 and 64")
    scenarios = []
    for depth_index, depth in enumerate(depths):
        if depth not in DEPTHS:
            raise ValueError(f"Unsupported target depth: {depth}")
        for index in range(scenarios_per_depth):
            scenario_seed = base_seed + depth_index * 100_000 + index
            rng = random.Random(scenario_seed)
            keys = _make_similar_keys(rng, record_count)
            codes = _make_unique_codes(rng, record_count)
            target_index = rng.randrange(record_count)
            records = [
                {"key": key, "code": code, "is_target": position == target_index}
                for position, (key, code) in enumerate(zip(keys, codes))
            ]
            rng.shuffle(records)
            target = next(record for record in records if record["is_target"])
            scenario = {
                "scenario_id": f"d{int(depth * 100):02d}-s{index:02d}",
                "seed": scenario_seed,
                "depth": depth,
                "target_key": target["key"],
                "target_code": target["code"],
                "records": records,
            }
            validate_semantic_scenario(scenario)
            scenarios.append(scenario)
    manifest = {
        "manifest_schema": MANIFEST_SCHEMA,
        "generator_version": GENERATOR_VERSION,
        "status": "DRAFT_UNTIL_C1W_C2_PILOT_PASSES",
        "base_seed": base_seed,
        "depths": list(depths),
        "scenarios_per_depth": scenarios_per_depth,
        "record_count": record_count,
        "scenarios": scenarios,
    }
    manifest["semantic_sha256"] = canonical_hash(manifest)
    return manifest


def _make_filler_text(rng, minimum_paragraphs=512):
    paragraphs = []
    topic_order = list(FILLER_TOPICS)
    method_order = list(FILLER_METHODS)
    outcome_order = list(FILLER_OUTCOMES)
    for index in range(minimum_paragraphs):
        if index % len(topic_order) == 0:
            rng.shuffle(topic_order)
        if index % len(method_order) == 0:
            rng.shuffle(method_order)
        if index % len(outcome_order) == 0:
            rng.shuffle(outcome_order)
        topic = topic_order[index % len(topic_order)]
        method = method_order[index % len(method_order)]
        outcome = outcome_order[index % len(outcome_order)]
        paragraphs.append(
            f"Memo {index + 1} discusses {topic}. The working group {method}. "
            f"Sample batch {2000 + index} was reviewed by {2 + index % 7} analysts. "
            f"{outcome}\n"
        )
    text = "\n".join(paragraphs)
    if CODE_PATTERN.search(text):
        raise AssertionError("Filler text accidentally contains a code-shaped value")
    return text


def _record_text(record):
    return (
        f"\n[Access record] System {record['key']} uses activation code "
        f"{record['code']}.\n"
    )


def _encode(tokenizer, text):
    return list(tokenizer.encode(text, add_special_tokens=False))


def _allocate_gaps(total, count, rng):
    minimum = MIN_RECORD_GAP_TOKENS * count
    if total < minimum:
        raise ValueError(
            f"Region has {total} filler tokens but needs at least {minimum} "
            "to keep records separated"
        )
    remaining = total - minimum
    gaps = [MIN_RECORD_GAP_TOKENS] * count
    order = list(range(count))
    rng.shuffle(order)
    for index in range(remaining):
        gaps[order[index % count]] += 1
    return gaps


def _pack_region(tokenizer, records, token_budget, rng):
    record_ids = [_encode(tokenizer, _record_text(record)) for record in records]
    filler_budget = token_budget - sum(map(len, record_ids))
    gaps = _allocate_gaps(filler_budget, len(records) + 1, rng)
    filler_text = _make_filler_text(rng)
    filler_ids = _encode(tokenizer, filler_text)
    while len(filler_ids) < filler_budget:
        filler_text += _make_filler_text(rng)
        filler_ids = _encode(tokenizer, filler_text)
    result = []
    cursor = 0
    for index, encoded_record in enumerate(record_ids):
        next_cursor = cursor + gaps[index]
        result.extend(filler_ids[cursor:next_cursor])
        result.extend(encoded_record)
        cursor = next_cursor
    result.extend(filler_ids[cursor:cursor + gaps[-1]])
    if len(result) != token_budget:
        raise AssertionError(f"Packed {len(result)} tokens, expected {token_budget}")
    return result, filler_text


def materialize_scenario(tokenizer, scenario, context_tokens=CONTEXT_TOKENS):
    validate_semantic_scenario(scenario)
    question = f"What is the code for {scenario['target_key']}?"
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {
            "role": "user",
            "content": f"<DOC>\n{PLACEHOLDER}\n</DOC>\n{question}",
        },
    ]
    template = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
    )
    if template.count(PLACEHOLDER) != 1:
        raise ValueError("Chat template did not preserve the document placeholder exactly once")
    prefix_text, suffix_text = template.split(PLACEHOLDER)
    prefix_ids = _encode(tokenizer, prefix_text)
    suffix_ids = _encode(tokenizer, suffix_text)
    if len(suffix_ids) > FINAL_QUERY_WINDOW:
        raise ValueError(
            f"Question/template suffix is {len(suffix_ids)} tokens; maximum is "
            f"{FINAL_QUERY_WINDOW}"
        )
    document_budget = context_tokens - len(prefix_ids) - len(suffix_ids)
    target_record = next(record for record in scenario["records"] if record["is_target"])
    distractors = [record for record in scenario["records"] if not record["is_target"]]
    target_ids = _encode(tokenizer, _record_text(target_record))
    target_start = round(scenario["depth"] * document_budget)
    after_budget = document_budget - target_start - len(target_ids)
    if target_start <= 0 or after_budget <= 0:
        raise ValueError("Target depth leaves no room on one side of the target record")
    rng = random.Random(scenario["seed"] + 17)
    rng.shuffle(distractors)
    before_count = round(scenario["depth"] * len(distractors))
    before_records = distractors[:before_count]
    after_records = distractors[before_count:]
    before_ids, before_filler = _pack_region(
        tokenizer, before_records, target_start, random.Random(scenario["seed"] + 31),
    )
    after_ids, after_filler = _pack_region(
        tokenizer, after_records, after_budget, random.Random(scenario["seed"] + 47),
    )
    document_ids = before_ids + target_ids + after_ids
    input_ids = prefix_ids + document_ids + suffix_ids
    if len(document_ids) != document_budget or len(input_ids) != context_tokens:
        raise AssertionError("Prompt materialization failed the exact-length gate")
    answer = scenario["target_code"]
    leak_text = SYSTEM_INSTRUCTION + suffix_text + before_filler + after_filler
    if answer.casefold() in leak_text.casefold():
        raise AssertionError("Target answer leaked outside its record")
    document_text = tokenizer.decode(document_ids, skip_special_tokens=False)
    extracted_document_codes = [code.upper() for code in CODE_PATTERN.findall(document_text)]
    expected_codes = {record["code"] for record in scenario["records"]}
    if len(extracted_document_codes) != len(scenario["records"]):
        raise AssertionError("A record was truncated or an extra code appeared in the document")
    if set(extracted_document_codes) != expected_codes:
        raise AssertionError("Materialized document codes differ from semantic records")
    if extracted_document_codes.count(answer) != 1:
        raise AssertionError("Target answer must occur exactly once in the document")
    realized_depth = target_start / document_budget
    if abs(realized_depth - scenario["depth"]) > 1 / document_budget:
        raise AssertionError("Target token depth exceeds one-token rounding tolerance")
    return {
        "scenario_id": scenario["scenario_id"],
        "depth": scenario["depth"],
        "target_key": scenario["target_key"],
        "target_code": answer,
        "record_count": len(scenario["records"]),
        "context_tokens": context_tokens,
        "document_tokens": document_budget,
        "target_start_document_token": target_start,
        "target_start_input_token": len(prefix_ids) + target_start,
        "target_record_tokens": len(target_ids),
        "realized_depth": realized_depth,
        "query_window_tokens": len(suffix_ids),
        "input_ids_sha256": token_ids_hash(input_ids),
        "input_ids": input_ids,
    }


def materialize_prompt_manifest(
    tokenizer,
    semantic_manifest,
    tokenizer_id,
    tokenizer_revision,
    context_tokens=CONTEXT_TOKENS,
    include_input_ids=True,
):
    unhashed = {key: value for key, value in semantic_manifest.items() if key != "semantic_sha256"}
    if semantic_manifest.get("semantic_sha256") != canonical_hash(unhashed):
        raise ValueError("Semantic manifest hash is missing or invalid")
    prompts = [
        materialize_scenario(tokenizer, scenario, context_tokens=context_tokens)
        for scenario in semantic_manifest["scenarios"]
    ]
    prompt_set_sha256 = canonical_hash(
        [
            {"scenario_id": prompt["scenario_id"], "sha256": prompt["input_ids_sha256"]}
            for prompt in prompts
        ]
    )
    if not include_input_ids:
        prompts = [
            {key: value for key, value in prompt.items() if key != "input_ids"}
            for prompt in prompts
        ]
    return {
        "manifest_schema": MANIFEST_SCHEMA,
        "generator_version": GENERATOR_VERSION,
        "status": semantic_manifest["status"],
        "semantic_sha256": semantic_manifest["semantic_sha256"],
        "prompt_set_sha256": prompt_set_sha256,
        "tokenizer_id": tokenizer_id,
        "tokenizer_revision": tokenizer_revision,
        "context_tokens": context_tokens,
        "query_window_limit": FINAL_QUERY_WINDOW,
        "prompts": prompts,
    }


def score_response(response, target_code):
    if CODE_PATTERN.fullmatch(target_code) is None:
        raise ValueError(f"Invalid target code: {target_code}")
    extracted = [code.upper() for code in CODE_PATTERN.findall(response)]
    unique_codes = list(dict.fromkeys(extracted))
    target = target_code.upper()
    retrieval_correct = len(unique_codes) == 1 and unique_codes[0] == target
    exact_format = response.strip() == target_code
    legacy_target_anywhere = target.casefold() in response.casefold()
    if retrieval_correct:
        failure_category = "pass"
    elif len(unique_codes) > 1:
        failure_category = "multiple_code"
    elif len(unique_codes) == 1:
        failure_category = "wrong_code"
    elif PARTIAL_CODE_PATTERN.search(response):
        failure_category = "truncated_code"
    else:
        failure_category = "no_code"
    return {
        "retrieval_correct": retrieval_correct,
        "format_correct": exact_format,
        "legacy_target_anywhere": legacy_target_anywhere,
        "failure_category": failure_category,
        "extracted_codes": extracted,
        "unique_extracted_codes": unique_codes,
    }


assert score_response("A123-K45", "A123-K45")["retrieval_correct"]
assert score_response("A123-K", "A123-K45")["failure_category"] == "truncated_code"
assert score_response("none", "A123-K45")["failure_category"] == "no_code"
assert not score_response("A123-K45 B234-L56", "A123-K45")["retrieval_correct"]
print("Frozen generator and strict scorer defined")


## 4. Independent Qwen Tokenization and Protocol Fingerprint

Both pinned repositories must expose the same tokenizer serialization, chat template, and special-token map. The 60 scenarios are independently materialized to exactly 4096 Qwen tokens. The Qwen prompt hash and protocol fingerprint were predeclared before any model output was inspected.


In [ ]:
def atomic_write_text(path, text):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + f".{os.getpid()}.tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)


def atomic_write_json(path, value):
    atomic_write_text(path, json.dumps(value, indent=2, ensure_ascii=True) + "\n")


def tokenizer_identity(active_tokenizer):
    return {
        "backend_sha256": hashlib.sha256(
            active_tokenizer.backend_tokenizer.to_str().encode("utf-8")
        ).hexdigest(),
        "chat_template_sha256": hashlib.sha256(
            active_tokenizer.chat_template.encode("utf-8")
        ).hexdigest(),
        "special_tokens_sha256": canonical_hash(active_tokenizer.special_tokens_map),
    }


def validate_config(config, quantized):
    assert config.model_type == "qwen2"
    assert config.architectures == ["Qwen2ForCausalLM"]
    assert config.num_hidden_layers == NUM_LAYERS
    assert config.hidden_size == HIDDEN_SIZE
    assert config.intermediate_size == INTERMEDIATE_SIZE
    assert config.num_attention_heads == NUM_Q_HEADS
    assert config.num_key_value_heads == NUM_KV_HEADS
    assert NUM_Q_HEADS % NUM_KV_HEADS == 0
    assert getattr(config, "head_dim", HIDDEN_SIZE // NUM_Q_HEADS) == HEAD_DIM
    assert config.vocab_size == VOCAB_SIZE
    assert config.max_position_embeddings == MAX_POSITIONS
    assert config.attention_dropout == 0.0
    assert config.tie_word_embeddings is True
    assert config.sliding_window is None
    assert list(config.layer_types) == ["full_attention"] * NUM_LAYERS
    assert float(config.rope_parameters["rope_theta"]) == ROPE_THETA
    if quantized:
        raw = config.quantization_config
        quant = raw.to_dict() if hasattr(raw, "to_dict") else dict(raw)
        assert quant["quant_method"] == "awq"
        assert quant["bits"] == 4 and quant["group_size"] == 128
        assert quant["zero_point"] is True
        awq_format = quant.get("format", quant.get("version"))
        assert str(awq_format).lower().endswith("gemm"), awq_format


tokenizer_fp16 = AutoTokenizer.from_pretrained(
    MODEL_FP16, revision=MODEL_FP16_REVISION, trust_remote_code=False,
)
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_AWQ, revision=MODEL_AWQ_REVISION, trust_remote_code=False,
)
fp16_tokenizer_identity = tokenizer_identity(tokenizer_fp16)
awq_tokenizer_identity = tokenizer_identity(tokenizer)
assert fp16_tokenizer_identity == awq_tokenizer_identity
assert awq_tokenizer_identity == {
    "backend_sha256": EXPECTED_TOKENIZER_BACKEND_SHA256,
    "chat_template_sha256": EXPECTED_CHAT_TEMPLATE_SHA256,
    "special_tokens_sha256": EXPECTED_SPECIAL_TOKENS_SHA256,
}
assert tokenizer.pad_token_id == 151643
assert tokenizer.eos_token_id == 151645
assert tokenizer.bos_token_id is None and tokenizer.unk_token_id is None
assert len(tokenizer) == 151665
assert max(tokenizer.get_vocab().values()) == 151664 < VOCAB_SIZE
del tokenizer_fp16

fp16_config = AutoConfig.from_pretrained(MODEL_FP16, revision=MODEL_FP16_REVISION)
awq_config = AutoConfig.from_pretrained(MODEL_AWQ, revision=MODEL_AWQ_REVISION)
validate_config(fp16_config, quantized=False)
validate_config(awq_config, quantized=True)

semantic_manifest = build_semantic_manifest()
expected_scenario_ids = [
    f"d{int(depth * 100):02d}-s{index:02d}"
    for depth in DEPTHS for index in range(SCENARIOS_PER_DEPTH)
]
assert [row["scenario_id"] for row in semantic_manifest["scenarios"]] == expected_scenario_ids
assert len(semantic_manifest["scenarios"]) == 60
assert semantic_manifest["semantic_sha256"] == EXPECTED_SEMANTIC_SHA256

prompt_manifest = materialize_prompt_manifest(
    tokenizer,
    semantic_manifest,
    tokenizer_id=MODEL_AWQ,
    tokenizer_revision=MODEL_AWQ_REVISION,
    context_tokens=CONTEXT_TOKENS,
    include_input_ids=True,
)
assert [row["scenario_id"] for row in prompt_manifest["prompts"]] == expected_scenario_ids
assert prompt_manifest["prompt_set_sha256"] == EXPECTED_PROMPT_SET_SHA256
assert all(len(row["input_ids"]) == CONTEXT_TOKENS for row in prompt_manifest["prompts"])
assert all(row["record_count"] == RECORD_COUNT for row in prompt_manifest["prompts"])
assert all(row["query_window_tokens"] <= FINAL_QUERY_WINDOW for row in prompt_manifest["prompts"])
assert {row["depth"] for row in prompt_manifest["prompts"]} == set(DEPTHS)
assert all(sum(row["depth"] == depth for row in prompt_manifest["prompts"]) == 20 for depth in DEPTHS)
prompts = {row["scenario_id"]: row for row in prompt_manifest["prompts"]}
assert len(prompts) == 60

def build_pyramid_budgets(layers=NUM_LAYERS, average_budget=DEFAULT_AVERAGE_BUDGET,
                          window=WINDOW, beta=PYRAMID_BETA):
    old_average = int(average_budget) - window
    last = round(old_average / beta)
    first = 2 * old_average - last
    budgets = (np.linspace(first, last, layers).round().astype(int) + window).tolist()
    assert len(budgets) == layers and sum(budgets) == layers * average_budget
    assert min(budgets) >= window and all(a >= b for a, b in zip(budgets, budgets[1:]))
    return budgets


PYRAMID_BUDGETS = build_pyramid_budgets()
assert PYRAMID_BUDGETS == [
    1966, 1912, 1858, 1805, 1751, 1697, 1643, 1589, 1535, 1482, 1428, 1374,
    1320, 1266, 1212, 1159, 1105, 1051, 997, 943, 889, 836, 782, 728, 674,
    620, 566, 513, 459, 405, 351, 297, 243, 190, 136, 82,
]
FINGERPRINT_DATA = {
    "schema_version": SCHEMA_VERSION,
    "model_fp16": MODEL_FP16,
    "model_fp16_revision": MODEL_FP16_REVISION,
    "model_awq": MODEL_AWQ,
    "model_awq_revision": MODEL_AWQ_REVISION,
    "transformers": "5.12.1",
    "transformers_commit": TRANSFORMERS_COMMIT,
    "qwen2_source_sha256": EXPECTED_QWEN2_SOURCE_SHA256,
    "cache_utils_source_sha256": EXPECTED_CACHE_UTILS_SOURCE_SHA256,
    "masking_utils_source_sha256": EXPECTED_MASKING_UTILS_SOURCE_SHA256,
    "sdpa_source_sha256": EXPECTED_SDPA_SOURCE_SHA256,
    "numpy": "2.2.6",
    "scipy": "1.16.1",
    "gptqmodel": "7.1.0",
    "turboquant": "0.2.0",
    "turboquant_source_sha256": EXPECTED_TQ_SOURCE_SHA256,
    "generator_version": GENERATOR_VERSION,
    "semantic_sha256": EXPECTED_SEMANTIC_SHA256,
    "prompt_set_sha256": EXPECTED_PROMPT_SET_SHA256,
    "tokenizer_backend_sha256": EXPECTED_TOKENIZER_BACKEND_SHA256,
    "chat_template_sha256": EXPECTED_CHAT_TEMPLATE_SHA256,
    "special_tokens_sha256": EXPECTED_SPECIAL_TOKENS_SHA256,
    "context_tokens": CONTEXT_TOKENS,
    "architecture": {
        "layers": NUM_LAYERS, "hidden_size": HIDDEN_SIZE,
        "intermediate_size": INTERMEDIATE_SIZE, "query_heads": NUM_Q_HEADS,
        "kv_heads": NUM_KV_HEADS, "head_dim": HEAD_DIM, "vocab_size": VOCAB_SIZE,
        "max_positions": MAX_POSITIONS, "rope_theta": ROPE_THETA,
    },
    "pyramid_budgets": PYRAMID_BUDGETS,
    "average_budget": DEFAULT_AVERAGE_BUDGET,
    "window": WINDOW,
    "pool_kernel": POOL_KERNEL,
    "pyramid_beta": PYRAMID_BETA,
    "tq_bits": DEFAULT_TQ_BITS,
    "tq_nmse_max": TQ_NMSE_MAX,
    "parity_atol": PARITY_ATOL,
    "parity_rtol": PARITY_RTOL,
    "parity_scenarios": list(PARITY_SCENARIOS),
    "probe_scenario": PROBE_SCENARIO,
    "probe_decode_tokens": 1,
    "variants": list(VARIANTS),
    "tq_ref_boundary": "not packed, fused, or physically 4-bit",
}
EXPERIMENT_FINGERPRINT = canonical_hash(FINGERPRINT_DATA)[:16]
assert EXPERIMENT_FINGERPRINT == EXPECTED_FINGERPRINT, (
    EXPERIMENT_FINGERPRINT, EXPECTED_FINGERPRINT
)

if SAVE_DIR.exists() and any(SAVE_DIR.iterdir()):
    raise RuntimeError(
        f"Pilot directory {SAVE_DIR} is nonempty. Preserve it and use a new protocol version; "
        "never mix attempts or overwrite pilot artifacts."
    )
SAVE_DIR.mkdir(parents=True, exist_ok=True)
environment = {
    **FINGERPRINT_DATA,
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "session_id": SESSION_ID,
    "artifact_path": SAVE_DIR.name,
    "gpu": GPU.name,
    "vram_gb": round(GPU.total_memory / 2**30, 2),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "tokenizer_identity": awq_tokenizer_identity,
    "qwen2_forward_parameters": list(forward_parameters),
    "status": "PROMPTS_FROZEN_VALIDATION_PENDING",
}
atomic_write_json(SEMANTIC_MANIFEST_JSON, semantic_manifest)
atomic_write_json(PROMPT_MANIFEST_JSON, prompt_manifest)
atomic_write_json(ENVIRONMENT_JSON, environment)
atomic_write_json(VALIDATION_JSON, {
    "passed": False,
    "schema_version": SCHEMA_VERSION,
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "status": "VALIDATION_INCOMPLETE",
})
print("Frozen Qwen prompt and protocol fingerprint passed:", EXPERIMENT_FINGERPRINT)


## 5. Qwen-Specific Wrapper, Cache Audit, and Model Gates

The wrapper retains the existing Q/K/V projections, including Qwen biases and packed AWQ modules. RoPE is applied exactly once before selection. Prefill attention always sees all 4096 tokens; only afterward does Q2/Q4-Ref select storage. Q4-Ref sends selected K/V into TQ-Ref, establishing eviction-first order. Decode feeds the exact tensors returned by `cache.update` into attention.


In [ ]:
def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def release_model():
    previous = globals().pop("model", None)
    if previous is not None:
        del previous
    clear_gpu()


def validate_model_architecture(model, quantized):
    validate_config(model.config, quantized=quantized)
    assert len(model.model.layers) == NUM_LAYERS
    assert model.model.embed_tokens.num_embeddings == VOCAB_SIZE
    for index, layer in enumerate(model.model.layers):
        attention = layer.self_attn
        assert isinstance(attention, Qwen2Attention)
        assert attention.layer_idx == index
        assert attention.num_key_value_groups == NUM_Q_HEADS // NUM_KV_HEADS == 8
        assert attention.head_dim == HEAD_DIM
        assert attention.layer_type == "full_attention"
        assert attention.sliding_window is None
        assert attention.q_proj.bias is not None
        assert attention.k_proj.bias is not None
        assert attention.v_proj.bias is not None
        assert attention.o_proj.bias is None
        if quantized:
            assert not isinstance(attention.q_proj, torch.nn.Linear)
            assert not isinstance(attention.k_proj, torch.nn.Linear)
            assert not isinstance(attention.v_proj, torch.nn.Linear)
            assert not isinstance(attention.o_proj, torch.nn.Linear)
    if not quantized:
        assert model.model.embed_tokens.weight.dtype == torch.float16
    assert next(model.parameters()).device.type == "cuda"


def load_fp16():
    assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
    loaded = AutoModelForCausalLM.from_pretrained(
        MODEL_FP16, revision=MODEL_FP16_REVISION, dtype=torch.float16,
        device_map={"": 0}, attn_implementation="sdpa", trust_remote_code=False,
    ).eval()
    validate_model_architecture(loaded, quantized=False)
    return loaded


def load_awq():
    assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
    loaded = AutoModelForCausalLM.from_pretrained(
        MODEL_AWQ, revision=MODEL_AWQ_REVISION, device_map={"": 0},
        attn_implementation="sdpa", trust_remote_code=False,
    ).eval()
    validate_model_architecture(loaded, quantized=True)
    return loaded


VARIANT_PATHS = {
    "Q0": {"wrapper": False, "eviction": False, "tq": False},
    "Q1-S": {"wrapper": False, "eviction": False, "tq": False},
    "Q1-W": {"wrapper": True, "eviction": False, "tq": False},
    "Q2": {"wrapper": True, "eviction": True, "tq": False},
    "Q3-Ref": {"wrapper": True, "eviction": False, "tq": True},
    "Q4-Ref": {"wrapper": True, "eviction": True, "tq": True},
}
PKV_LOG = []
CACHE_UPDATE_LOG = []
POSITION_LOG = []
EVENT_LOG = []
ATTENTION_CALL_LOG = []
TQ_PATH_LOG = []
ACTIVE_EVICTION = False
ORIGINAL_QWEN_FORWARD = Qwen2Attention.forward


def select_pyramid_kv(query_states, key_states, value_states, layer_idx, groups, scaling):
    cache_len = key_states.shape[-2]
    budget = min(PYRAMID_BUDGETS[layer_idx], cache_len)
    assert cache_len == CONTEXT_TOKENS and cache_len > budget
    obs = min(WINDOW, cache_len)
    old_len = cache_len - obs
    old_keep = budget - obs
    recent_query = query_states[:, :, -obs:, :]
    repeated_key = repeat_kv(key_states, groups)
    scores = torch.matmul(recent_query, repeated_key.transpose(2, 3)) * scaling
    causal = torch.triu(
        torch.full((obs, obs), float("-inf"), device=scores.device, dtype=scores.dtype),
        diagonal=1,
    )
    scores[:, :, :, -obs:] += causal[None, None, :, :]
    probabilities = F.softmax(scores, dim=-1, dtype=torch.float32)
    prefix_scores = probabilities[:, :, :, :old_len].sum(dim=-2)
    prefix_scores = prefix_scores.reshape(
        prefix_scores.shape[0], NUM_KV_HEADS, groups, old_len,
    ).mean(dim=2)
    prefix_scores = F.avg_pool1d(
        prefix_scores, kernel_size=POOL_KERNEL, stride=1, padding=POOL_KERNEL // 2,
    )
    topk = prefix_scores.topk(old_keep, dim=-1).indices.sort(dim=-1).values
    gather_index = topk.unsqueeze(-1).expand(-1, -1, -1, HEAD_DIM)
    selected_key = torch.cat([
        key_states[:, :, :old_len, :].gather(2, gather_index),
        key_states[:, :, -obs:, :],
    ], dim=2).contiguous()
    selected_value = torch.cat([
        value_states[:, :, :old_len, :].gather(2, gather_index),
        value_states[:, :, -obs:, :],
    ], dim=2).contiguous()
    PKV_LOG.append({
        "layer": int(layer_idx), "before": int(cache_len),
        "after": int(selected_key.shape[-2]), "budget": int(budget),
        "head_selection_differs": bool((topk[:, 1:] != topk[:, :-1]).any().item()),
    })
    EVENT_LOG.append({"phase": "prefill", "layer": int(layer_idx), "event": "eviction"})
    return selected_key, selected_value


def audited_attention_call(interface, module, query, key, value, attention_mask,
                           expected_key_ptr, expected_value_ptr, expected_mask_id, **kwargs):
    observed_key_ptr = int(key.data_ptr())
    observed_value_ptr = int(value.data_ptr())
    assert id(attention_mask) == expected_mask_id
    if expected_key_ptr is not None:
        assert observed_key_ptr == expected_key_ptr
        assert observed_value_ptr == expected_value_ptr
    ATTENTION_CALL_LOG.append({
        "layer": int(module.layer_idx),
        "query_tokens": int(query.shape[-2]),
        "key_tokens": int(key.shape[-2]),
        "observed_key_ptr": observed_key_ptr,
        "observed_value_ptr": observed_value_ptr,
        "expected_key_ptr": expected_key_ptr,
        "expected_value_ptr": expected_value_ptr,
        "mask_object_id": id(attention_mask),
        "expected_mask_id": expected_mask_id,
        "mask_identity_preserved": id(attention_mask) == expected_mask_id,
    })
    return interface(module, query, key, value, attention_mask, **kwargs)


def qwen_cache_wrapper_forward(self, hidden_states, position_embeddings,
                               attention_mask, past_key_values=None, **kwargs):
    assert self.layer_type == "full_attention" and self.sliding_window is None
    input_shape = hidden_states.shape[:-1]
    hidden_shape = (*input_shape, -1, self.head_dim)
    query_states = self.q_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    key_states = self.k_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    value_states = self.v_proj(hidden_states).view(hidden_shape).transpose(1, 2)
    assert query_states.shape[1] == NUM_Q_HEADS
    assert key_states.shape[1] == value_states.shape[1] == NUM_KV_HEADS
    cos, sin = position_embeddings
    query_states, key_states = apply_rotary_pos_emb(query_states, key_states, cos, sin)

    position_ids = kwargs.get("position_ids")
    assert position_ids is not None
    input_tokens = int(key_states.shape[-2])
    assert tuple(position_ids.shape) == (1, input_tokens)
    position_start = int(position_ids[0, 0])
    position_end = int(position_ids[0, -1])
    assert position_end - position_start + 1 == input_tokens
    is_prefill = (
        past_key_values is not None
        and past_key_values.get_seq_length(self.layer_idx) == 0
        and input_tokens == CONTEXT_TOKENS
    )
    phase = "prefill" if is_prefill else "decode"
    if is_prefill:
        assert position_start == 0 and position_end == CONTEXT_TOKENS - 1
    else:
        assert input_tokens == 1 and position_start >= CONTEXT_TOKENS
        assert attention_mask is None, "Single-token decode must use the explicit unpadded mask bypass"
    POSITION_LOG.append({
        "phase": phase, "layer": int(self.layer_idx),
        "position_start": position_start, "position_end": position_end,
        "input_tokens": input_tokens,
    })

    expected_attention_key_ptr = None
    expected_attention_value_ptr = None
    if past_key_values is not None and not is_prefill:
        returned_key, returned_value = past_key_values.update(
            key_states, value_states, self.layer_idx,
        )
        key_states, value_states = returned_key, returned_value
        expected_attention_key_ptr = int(returned_key.data_ptr())
        expected_attention_value_ptr = int(returned_value.data_ptr())
        CACHE_UPDATE_LOG.append({
            "phase": "decode", "layer": int(self.layer_idx),
            "input_tokens": input_tokens, "returned_tokens": int(key_states.shape[-2]),
            "position": position_start, "cache_type": type(past_key_values).__name__,
            "returned_key_is_attention_key": returned_key is key_states,
            "returned_value_is_attention_value": returned_value is value_states,
            "returned_key_data_ptr": int(returned_key.data_ptr()),
            "attention_key_data_ptr": int(key_states.data_ptr()),
        })
        EVENT_LOG.append({"phase": "decode", "layer": int(self.layer_idx), "event": "cache_update"})

    attention_tokens = int(key_states.shape[-2])
    mask_object = attention_mask
    attention_interface = ALL_ATTENTION_FUNCTIONS.get_interface(
        self.config._attn_implementation, eager_attention_forward,
    )
    attention_output, attention_weights = audited_attention_call(
        attention_interface, self, query_states, key_states, value_states, attention_mask,
        expected_key_ptr=expected_attention_key_ptr,
        expected_value_ptr=expected_attention_value_ptr,
        expected_mask_id=id(mask_object),
        dropout=0.0 if not self.training else self.attention_dropout,
        scaling=self.scaling, sliding_window=self.sliding_window, **kwargs,
    )
    assert mask_object is attention_mask
    EVENT_LOG.append({"phase": phase, "layer": int(self.layer_idx), "event": "attention"})

    if is_prefill:
        assert attention_tokens == CONTEXT_TOKENS
        stored_key, stored_value = key_states, value_states
        if ACTIVE_EVICTION:
            stored_key, stored_value = select_pyramid_kv(
                query_states, key_states, value_states, self.layer_idx,
                self.num_key_value_groups, self.scaling,
            )
        update_input_tokens = int(stored_key.shape[-2])
        returned_key, returned_value = past_key_values.update(
            stored_key, stored_value, self.layer_idx,
        )
        EVENT_LOG.append({"phase": "prefill", "layer": int(self.layer_idx), "event": "cache_update"})
        if isinstance(past_key_values, TurboQuantCache):
            quantized_tokens = max(update_input_tokens - 128, 0)
            if quantized_tokens:
                key_nmse = float(
                    ((returned_key[..., :quantized_tokens, :].float() - stored_key[..., :quantized_tokens, :].float()).square().sum(-1)
                     / stored_key[..., :quantized_tokens, :].float().square().sum(-1).clamp_min(1e-12)).mean()
                )
                value_nmse = float(
                    ((returned_value[..., :quantized_tokens, :].float() - stored_value[..., :quantized_tokens, :].float()).square().sum(-1)
                     / stored_value[..., :quantized_tokens, :].float().square().sum(-1).clamp_min(1e-12)).mean()
                )
            else:
                key_nmse = value_nmse = 0.0
            residual_tokens = min(update_input_tokens, 128)
            residual_keys_exact = torch.equal(
                returned_key[..., -residual_tokens:, :], stored_key[..., -residual_tokens:, :]
            )
            residual_values_exact = torch.equal(
                returned_value[..., -residual_tokens:, :], stored_value[..., -residual_tokens:, :]
            )
            assert key_nmse <= TQ_NMSE_MAX and value_nmse <= TQ_NMSE_MAX
            assert residual_keys_exact and residual_values_exact
            TQ_PATH_LOG.append({
                "layer": int(self.layer_idx),
                "input_tokens": update_input_tokens,
                "quantized_tokens": quantized_tokens,
                "key_nmse": key_nmse,
                "value_nmse": value_nmse,
                "residual_keys_exact": residual_keys_exact,
                "residual_values_exact": residual_values_exact,
            })
        CACHE_UPDATE_LOG.append({
            "phase": "prefill", "layer": int(self.layer_idx),
            "attention_tokens": attention_tokens,
            "update_input_tokens": update_input_tokens,
            "returned_tokens": int(returned_key.shape[-2]),
            "cache_type": type(past_key_values).__name__,
            "mask_identity_preserved": mask_object is attention_mask,
            "rope_before_storage": True,
        })
        del returned_key, returned_value

    attention_output = attention_output.reshape(*input_shape, -1).contiguous()
    return self.o_proj(attention_output), attention_weights


def enable_wrapper(use_eviction):
    global ACTIVE_EVICTION
    assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
    PKV_LOG.clear(); CACHE_UPDATE_LOG.clear(); POSITION_LOG.clear(); EVENT_LOG.clear()
    ATTENTION_CALL_LOG.clear(); TQ_PATH_LOG.clear()
    ACTIVE_EVICTION = bool(use_eviction)
    Qwen2Attention.forward = qwen_cache_wrapper_forward


def disable_wrapper():
    global ACTIVE_EVICTION
    Qwen2Attention.forward = ORIGINAL_QWEN_FORWARD
    ACTIVE_EVICTION = False
    assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD


def new_cache(model, variant):
    if VARIANT_PATHS[variant]["tq"]:
        return TurboQuantCache(bits=DEFAULT_TQ_BITS)
    return DynamicCache(config=model.config)


def cache_lengths(cache):
    return [
        int(layer._total_len) if hasattr(layer, "_total_len") else int(layer.get_seq_length())
        for layer in cache.layers
    ]


def named_cache_tensors(cache):
    for layer_index, layer in enumerate(cache.layers):
        for name, value in vars(layer).items():
            if isinstance(value, torch.Tensor):
                yield layer_index, name, value
            elif isinstance(value, Mapping):
                for nested_name, nested_value in value.items():
                    if isinstance(nested_value, torch.Tensor):
                        yield layer_index, f"{name}.{nested_name}", nested_value
            elif isinstance(value, (list, tuple)):
                for nested_index, nested_value in enumerate(value):
                    if isinstance(nested_value, torch.Tensor):
                        yield layer_index, f"{name}.{nested_index}", nested_value


def unique_storage_summary(tensors):
    unique = {}
    references = 0
    for tensor in tensors:
        if not isinstance(tensor, torch.Tensor) or tensor.numel() == 0:
            continue
        references += 1
        storage = tensor.untyped_storage()
        key = (str(tensor.device), int(storage.data_ptr()), int(storage.nbytes()))
        unique[key] = int(storage.nbytes())
    return {
        "bytes": sum(unique.values()),
        "unique_storages": len(unique),
        "tensor_references": references,
        "aliased_references": references - len(unique),
    }


def cache_storage_metrics(cache, prefix):
    named = list(named_cache_tensors(cache))
    all_tensors = [tensor for _, _, tensor in named]
    indices = [tensor for _, name, tensor in named if name in {"_key_indices", "_value_indices"}]
    norms = [tensor for _, name, tensor in named if name in {"_key_norms", "_value_norms"}]
    residuals = [tensor for _, name, tensor in named if name in {"_residual_keys", "_residual_values"}]
    materialized = [tensor for _, name, tensor in named if name in {"keys", "values"}]
    all_summary = unique_storage_summary(all_tensors)
    raw_reference_bytes = sum(
        int(tensor.untyped_storage().nbytes()) for tensor in all_tensors if tensor.numel() > 0
    )
    logical_payload_bytes = sum(
        tensor.numel() * tensor.element_size()
        for tensor in indices + norms + residuals
    )
    package_bytes = 0
    if callable(getattr(cache, "memory_usage_bytes", None)):
        package_bytes = int(cache.memory_usage_bytes().get("total_bytes", 0))
    package_reconciled = package_bytes == logical_payload_bytes
    return {
        f"{prefix}_cache_live_bytes": all_summary["bytes"],
        f"{prefix}_unique_storages": all_summary["unique_storages"],
        f"{prefix}_tensor_references": all_summary["tensor_references"],
        f"{prefix}_aliased_references": all_summary["aliased_references"],
        f"{prefix}_raw_reference_bytes": raw_reference_bytes,
        f"{prefix}_alias_deduplicated": (
            all_summary["bytes"] <= raw_reference_bytes and package_reconciled
        ),
        f"{prefix}_logical_tq_payload_bytes": logical_payload_bytes,
        f"{prefix}_package_bytes_reconciled": package_reconciled,
        f"{prefix}_tq_index_bytes": unique_storage_summary(indices)["bytes"],
        f"{prefix}_tq_norm_bytes": unique_storage_summary(norms)["bytes"],
        f"{prefix}_tq_residual_bytes": unique_storage_summary(residuals)["bytes"],
        f"{prefix}_tq_materialized_bytes": unique_storage_summary(materialized)["bytes"],
        f"{prefix}_tq_package_reported_bytes": package_bytes,
    }


TQ_STATE_NAMES = {
    "_key_indices", "_value_indices", "_key_norms", "_value_norms",
    "_residual_keys", "_residual_values",
}


def assert_no_tq_state(cache):
    assert isinstance(cache, DynamicCache) and not isinstance(cache, TurboQuantCache)
    assert all(name not in TQ_STATE_NAMES for _, name, _ in named_cache_tensors(cache))


def validate_cache_path(cache, variant, prefill_lengths, decode_tokens,
                        update_log, position_log, event_log, pkv_log,
                        attention_call_log, tq_path_log):
    path = VARIANT_PATHS[variant]
    assert len(cache.layers) == NUM_LAYERS
    expected_prefill = PYRAMID_BUDGETS if path["eviction"] else [CONTEXT_TOKENS] * NUM_LAYERS
    assert prefill_lengths == expected_prefill
    final_lengths = cache_lengths(cache)
    assert final_lengths == [length + decode_tokens for length in expected_prefill]
    prefill_updates = [row for row in update_log if row["phase"] == "prefill"]
    decode_updates = [row for row in update_log if row["phase"] == "decode"]
    if path["wrapper"]:
        assert len(prefill_updates) == NUM_LAYERS
        assert all(row["attention_tokens"] == CONTEXT_TOKENS for row in prefill_updates)
        assert all(row["mask_identity_preserved"] and row["rope_before_storage"] for row in prefill_updates)
        assert [row["update_input_tokens"] for row in prefill_updates] == expected_prefill
        assert len(decode_updates) == NUM_LAYERS * decode_tokens
        decode_attention_calls = [row for row in attention_call_log if row["query_tokens"] == 1]
        assert len(decode_attention_calls) == NUM_LAYERS * decode_tokens
        assert all(
            row["observed_key_ptr"] == row["expected_key_ptr"]
            and row["observed_value_ptr"] == row["expected_value_ptr"]
            and row["mask_identity_preserved"]
            for row in decode_attention_calls
        )
        prefill_positions = [row for row in position_log if row["phase"] == "prefill"]
        assert len(prefill_positions) == NUM_LAYERS
        assert all(row["position_start"] == 0 and row["position_end"] == 4095 for row in prefill_positions)
        for layer in range(NUM_LAYERS):
            layer_decode = [row for row in decode_updates if row["layer"] == layer]
            assert [row["position"] for row in layer_decode] == list(
                range(CONTEXT_TOKENS, CONTEXT_TOKENS + decode_tokens)
            )
            layer_events = [
                row["event"] for row in event_log
                if row["phase"] == "prefill" and row["layer"] == layer
            ]
            expected_events = (
                ["attention", "eviction", "cache_update"]
                if path["eviction"] else ["attention", "cache_update"]
            )
            assert layer_events == expected_events
    else:
        assert not update_log and not position_log and not event_log and not pkv_log
        assert not attention_call_log and not tq_path_log
    if path["eviction"]:
        assert len(pkv_log) == NUM_LAYERS
        assert [row["layer"] for row in pkv_log] == list(range(NUM_LAYERS))
        assert all(row["before"] == CONTEXT_TOKENS for row in pkv_log)
        assert [row["after"] for row in pkv_log] == expected_prefill
    else:
        assert not pkv_log

    quantized_layers = 0
    residual_only_layers = []
    tq_actual_path_max_nmse = 0.0
    if path["tq"]:
        assert isinstance(cache, TurboQuantCache) and cache.bits == DEFAULT_TQ_BITS
        assert len(tq_path_log) == NUM_LAYERS
        assert [row["layer"] for row in tq_path_log] == list(range(NUM_LAYERS))
        assert all(row["residual_keys_exact"] and row["residual_values_exact"] for row in tq_path_log)
        tq_actual_path_max_nmse = max(
            max(row["key_nmse"], row["value_nmse"]) for row in tq_path_log
        )
        assert tq_actual_path_max_nmse <= TQ_NMSE_MAX
        for layer_index, (layer, total_length) in enumerate(zip(cache.layers, final_lengths)):
            assert layer.bits == DEFAULT_TQ_BITS and layer.residual_len == 128
            quantized_tokens = max(total_length - layer.residual_len, 0)
            residual_tokens = min(total_length, layer.residual_len)
            index_shape = (1, NUM_KV_HEADS, quantized_tokens, HEAD_DIM)
            norm_shape = (1, NUM_KV_HEADS, quantized_tokens, 1)
            if quantized_tokens:
                quantized_layers += 1
                assert tuple(layer._key_indices.shape) == index_shape
                assert tuple(layer._value_indices.shape) == index_shape
                assert tuple(layer._key_norms.shape) == norm_shape
                assert tuple(layer._value_norms.shape) == norm_shape
                for indices in (layer._key_indices, layer._value_indices):
                    assert indices.dtype == torch.uint8 and int(indices.max()) < 2 ** DEFAULT_TQ_BITS
                for norms in (layer._key_norms, layer._value_norms):
                    assert torch.isfinite(norms).all() and (norms >= 0).all()
            else:
                residual_only_layers.append(layer_index)
                assert layer._key_indices.numel() == layer._value_indices.numel() == 0
            assert layer._residual_keys.shape[-2] == residual_tokens
            assert layer._residual_values.shape[-2] == residual_tokens
            assert layer.keys.shape[-2] == total_length
            assert layer.values.shape[-2] == total_length
            assert torch.isfinite(layer.keys).all() and torch.isfinite(layer.values).all()
        assert quantized_layers == sum(length > 128 for length in final_lengths)
        if variant == "Q4-Ref":
            assert residual_only_layers == [NUM_LAYERS - 1]
        metrics = cache_storage_metrics(cache, "validation")
        assert metrics["validation_tq_index_bytes"] > 0
        assert metrics["validation_tq_norm_bytes"] > 0
        assert metrics["validation_tq_residual_bytes"] > 0
        assert metrics["validation_tq_materialized_bytes"] > 0
        assert metrics["validation_cache_live_bytes"] >= metrics["validation_tq_materialized_bytes"]
        assert metrics["validation_package_bytes_reconciled"]
        if variant == "Q4-Ref":
            assert metrics["validation_aliased_references"] >= 2
    else:
        assert_no_tq_state(cache)
    return {
        "structural_pass": True,
        "quantized_layers": quantized_layers,
        "residual_only_layers": residual_only_layers,
        "evicted_layers": NUM_LAYERS if path["eviction"] else 0,
        "cache_min_prefill": min(prefill_lengths),
        "cache_max_prefill": max(prefill_lengths),
        "cache_min_after": min(final_lengths),
        "cache_max_after": max(final_lengths),
        "active_decode_cache_updates": len(decode_updates),
        "active_attention_boundary_checks": sum(
            row["query_tokens"] == 1 for row in attention_call_log
        ),
        "tq_actual_path_max_nmse": tq_actual_path_max_nmse,
    }


print("Qwen model gates, wrapper, eviction scorer, and cache audit ready")


## 6. Q0 and AWQ Architecture Phases

Q0 is loaded only to prove the pinned FP16 architecture on the target T4, then released. The AWQ checkpoint is loaded separately and must retain Qwen2Attention while replacing all attention projections with the supported GPTQModel AWQ GEMM modules.


In [ ]:
model = load_fp16()
fp16_architecture = {
    "passed": True,
    "model_id": MODEL_FP16,
    "revision": MODEL_FP16_REVISION,
    "dtype": str(model.model.embed_tokens.weight.dtype),
    "layers": len(model.model.layers),
    "attention_class": type(model.model.layers[0].self_attn).__name__,
}
release_model()

model = load_awq()
awq_architecture = {
    "passed": True,
    "model_id": MODEL_AWQ,
    "revision": MODEL_AWQ_REVISION,
    "layers": len(model.model.layers),
    "attention_class": type(model.model.layers[0].self_attn).__name__,
    "q_projection_class": type(model.model.layers[0].self_attn.q_proj).__name__,
    "quantization": "AWQ GEMM 4-bit, group size 128, asymmetric zero point",
}
print("Q0 and AWQ architecture gates passed")


## 7. Q1-S/Q1-W Parity and Active-Cache Probes

The predeclared tolerances are `atol=0.005` and `rtol=0.005`. Three fixed prompts test final-position logits, top-1 agreement, exact greedy token IDs and decoded responses, all 36 layer lengths, absolute positions, and absence of compression state. One fixed prompt then performs a one-token cache-backed continuation for Q1-W, Q2, Q3-Ref, and Q4-Ref.


In [ ]:
def prompt_tensor(prompt, device):
    ids = torch.tensor([prompt["input_ids"]], dtype=torch.long, device=device)
    assert ids.shape == (1, CONTEXT_TOKENS)
    assert token_ids_hash(ids[0].detach().cpu().tolist()) == prompt["input_ids_sha256"]
    return ids


def snapshot_logs():
    return {
        "updates": list(CACHE_UPDATE_LOG),
        "positions": list(POSITION_LOG),
        "events": list(EVENT_LOG),
        "pkv": list(PKV_LOG),
        "attention_calls": list(ATTENTION_CALL_LOG),
        "tq_path": list(TQ_PATH_LOG),
    }


def clear_logs():
    PKV_LOG.clear(); CACHE_UPDATE_LOG.clear(); POSITION_LOG.clear(); EVENT_LOG.clear()
    ATTENTION_CALL_LOG.clear(); TQ_PATH_LOG.clear()


def run_greedy_path(model, variant, prompt, max_new_tokens=MAX_NEW_TOKENS):
    path = VARIANT_PATHS[variant]
    clear_logs()
    if path["wrapper"]:
        enable_wrapper(path["eviction"])
    else:
        assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
    try:
        cache = new_cache(model, variant)
        ids = prompt_tensor(prompt, model.device)
        prefill_positions = torch.arange(CONTEXT_TOKENS, device=model.device).unsqueeze(0)
    except Exception:
        if path["wrapper"] and Qwen2Attention.forward is not ORIGINAL_QWEN_FORWARD:
            disable_wrapper()
        raise
    generated = []
    try:
        with torch.inference_mode():
            output = model(
                input_ids=ids, position_ids=prefill_positions,
                past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
            prefill_lengths = cache_lengths(cache)
            prefill_logits = output.logits[0, -1].float().detach().cpu()
            next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
            stop_ids = set(np.atleast_1d(model.generation_config.eos_token_id).tolist())
            for step in range(max_new_tokens):
                generated.append(next_token)
                if int(next_token.item()) in stop_ids or step == max_new_tokens - 1:
                    break
                absolute_position = CONTEXT_TOKENS + step
                position = torch.tensor([[absolute_position]], device=model.device)
                output = model(
                    input_ids=next_token, position_ids=position,
                    attention_mask={"full_attention": None},
                    past_key_values=cache, use_cache=True, logits_to_keep=1,
                )
                next_token = output.logits[:, -1, :].argmax(-1, keepdim=True)
        output_ids = torch.cat(generated, dim=1)[0].tolist()
        logs = snapshot_logs()
        structure = validate_cache_path(
            cache, variant, prefill_lengths, max(0, len(output_ids) - 1),
            logs["updates"], logs["positions"], logs["events"], logs["pkv"],
            logs["attention_calls"], logs["tq_path"],
        )
        return {
            "prefill_logits": prefill_logits,
            "prefill_top1": int(prefill_logits.argmax()),
            "response": tokenizer.decode(output_ids, skip_special_tokens=True),
            "output_token_ids": output_ids,
            "prefill_lengths": prefill_lengths,
            "final_lengths": cache_lengths(cache),
            "structure": structure,
            "storage": cache_storage_metrics(cache, "generation_final"),
        }
    finally:
        if path["wrapper"]:
            disable_wrapper()
        del cache, ids
        clear_logs()
        clear_gpu()


def run_path_probe(model, variant, prompt):
    path = VARIANT_PATHS[variant]
    assert path["wrapper"]
    enable_wrapper(path["eviction"])
    try:
        cache = new_cache(model, variant)
        ids = prompt_tensor(prompt, model.device)
        answer_ids = tokenizer.encode(prompt["target_code"], add_special_tokens=False)
        assert tokenizer.decode(answer_ids, skip_special_tokens=True) == prompt["target_code"]
        prefill_positions = torch.arange(CONTEXT_TOKENS, device=model.device).unsqueeze(0)
    except Exception:
        if Qwen2Attention.forward is not ORIGINAL_QWEN_FORWARD:
            disable_wrapper()
        raise
    try:
        with torch.inference_mode():
            output = model(
                input_ids=ids, position_ids=prefill_positions,
                past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
            prefill_lengths = cache_lengths(cache)
            prefill_logits = output.logits[0, -1].float().detach().cpu()
            token = torch.tensor([[answer_ids[0]]], dtype=torch.long, device=model.device)
            position = torch.tensor([[CONTEXT_TOKENS]], device=model.device)
            model(
                input_ids=token, position_ids=position,
                attention_mask={"full_attention": None},
                past_key_values=cache, use_cache=True, logits_to_keep=1,
            )
        logs = snapshot_logs()
        structure = validate_cache_path(
            cache, variant, prefill_lengths, 1,
            logs["updates"], logs["positions"], logs["events"], logs["pkv"],
            logs["attention_calls"], logs["tq_path"],
        )
        return {
            "variant": variant,
            "input_ids_sha256": prompt["input_ids_sha256"],
            "prefill_logits": prefill_logits,
            "prefill_top1": int(prefill_logits.argmax()),
            "prefill_lengths": prefill_lengths,
            "final_lengths": cache_lengths(cache),
            "structure": structure,
            "storage": cache_storage_metrics(cache, "probe_final"),
            "prefill_update_inputs": [
                row["update_input_tokens"] for row in logs["updates"]
                if row["phase"] == "prefill"
            ],
        }
    finally:
        disable_wrapper()
        del cache, ids
        clear_logs()
        clear_gpu()


parity_rows = []
for scenario_id in PARITY_SCENARIOS:
    prompt = prompts[scenario_id]
    stock = run_greedy_path(model, "Q1-S", prompt)
    wrapped = run_greedy_path(model, "Q1-W", prompt)
    logits_allclose = torch.allclose(
        stock["prefill_logits"], wrapped["prefill_logits"],
        atol=PARITY_ATOL, rtol=PARITY_RTOL,
    )
    max_abs = float((stock["prefill_logits"] - wrapped["prefill_logits"]).abs().max())
    top1_agree = stock["prefill_top1"] == wrapped["prefill_top1"]
    token_ids_equal = stock["output_token_ids"] == wrapped["output_token_ids"]
    responses_equal = stock["response"] == wrapped["response"]
    cache_lengths_equal = (
        stock["prefill_lengths"] == wrapped["prefill_lengths"] == [CONTEXT_TOKENS] * NUM_LAYERS
        and stock["final_lengths"] == wrapped["final_lengths"]
    )
    assert logits_allclose and top1_agree and token_ids_equal and responses_equal
    assert cache_lengths_equal
    assert stock["structure"]["quantized_layers"] == 0
    assert wrapped["structure"]["quantized_layers"] == 0
    parity_rows.append({
        "scenario_id": scenario_id,
        "input_ids_sha256": prompt["input_ids_sha256"],
        "max_abs_logit_difference": max_abs,
        "logits_allclose": logits_allclose,
        "top1_agreement": top1_agree,
        "greedy_token_ids_equal": token_ids_equal,
        "greedy_responses_equal": responses_equal,
        "full_layer_cache_lengths_equal": cache_lengths_equal,
        "absence_of_eviction_tq_state": True,
        "stock_output_token_ids": stock["output_token_ids"],
        "wrapped_output_token_ids": wrapped["output_token_ids"],
        "stock_response": stock["response"],
        "wrapped_response": wrapped["response"],
    })

path_rows = []
probe_prompt = prompts[PROBE_SCENARIO]
for variant in ("Q1-W", "Q2", "Q3-Ref", "Q4-Ref"):
    path_rows.append(run_path_probe(model, variant, probe_prompt))
baseline_logits = path_rows[0]["prefill_logits"]
prefill_agreement = []
for probe in path_rows[1:]:
    max_abs = float((baseline_logits - probe["prefill_logits"]).abs().max())
    allclose = torch.allclose(
        baseline_logits, probe["prefill_logits"],
        atol=PARITY_ATOL, rtol=PARITY_RTOL,
    )
    top1_agree = probe["prefill_top1"] == path_rows[0]["prefill_top1"]
    assert allclose and top1_agree
    assert probe["structure"]["active_decode_cache_updates"] == NUM_LAYERS
    prefill_agreement.append({
        "variant": probe["variant"], "max_abs_logit_difference": max_abs,
        "logits_allclose": allclose, "top1_agreement": top1_agree,
    })
assert path_rows[1]["prefill_update_inputs"] == PYRAMID_BUDGETS
assert path_rows[3]["prefill_update_inputs"] == PYRAMID_BUDGETS
assert path_rows[2]["storage"]["probe_final_tq_materialized_bytes"] > 0
assert path_rows[3]["storage"]["probe_final_tq_materialized_bytes"] > 0
assert path_rows[3]["structure"]["residual_only_layers"] == [NUM_LAYERS - 1]

samples = torch.randn(1024, HEAD_DIM, device=model.device, dtype=torch.float16)
quantizer = TurboQuantMSE(dim=HEAD_DIM, bits=DEFAULT_TQ_BITS, device=str(model.device))
indices, norms = quantizer.quantize(samples)
restored = quantizer.dequantize(indices, norms)
normalized_mse = float(
    ((restored.float() - samples.float()).square().sum(-1)
     / samples.float().square().sum(-1).clamp_min(1e-12)).mean()
)
assert indices.dtype == torch.uint8 and int(indices.max()) < 2 ** DEFAULT_TQ_BITS
assert torch.isfinite(norms).all() and (norms >= 0).all()
assert torch.isfinite(restored).all()
assert 0.0 <= normalized_mse <= TQ_NMSE_MAX
assert indices.element_size() == 1
tq_numerical_probe = {
    "bits": DEFAULT_TQ_BITS,
    "normalized_mse": normalized_mse,
    "predeclared_nmse_max": TQ_NMSE_MAX,
    "physical_index_bytes_per_coordinate": indices.element_size(),
    "boundary": "numerically active reference path; not packed, fused, or physically 4-bit",
}

parity_record = {
    "passed": True,
    "schema_version": SCHEMA_VERSION,
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "atol": PARITY_ATOL,
    "rtol": PARITY_RTOL,
    "rows": parity_rows,
}
atomic_write_json(PARITY_JSON, parity_record)
serializable_paths = [
    {key: value for key, value in probe.items() if key != "prefill_logits"}
    for probe in path_rows
]
validation = {
    "passed": True,
    "schema_version": SCHEMA_VERSION,
    "fingerprint": EXPERIMENT_FINGERPRINT,
    "session_id": SESSION_ID,
    "artifact_path": SAVE_DIR.name,
    "semantic_sha256": EXPECTED_SEMANTIC_SHA256,
    "prompt_set_sha256": EXPECTED_PROMPT_SET_SHA256,
    "model_revisions": {"fp16": MODEL_FP16_REVISION, "awq": MODEL_AWQ_REVISION},
    "source_gates": {
        "transformers_commit": TRANSFORMERS_COMMIT,
        "qwen2_source_sha256": qwen2_source_sha256,
        "dependent_source_sha256": dependent_source_hashes,
        "qwen2_forward_parameters": list(forward_parameters),
        "cache_position_absent": "cache_position" not in forward_parameters,
        "turboquant_source_sha256": tq_source_sha256,
    },
    "architecture": {"fp16": fp16_architecture, "awq": awq_architecture},
    "prompt_gate_passed": True,
    "parity_gate_passed": all(row["logits_allclose"] for row in parity_rows),
    "position_gate_passed": True,
    "cache_gate_passed": all(row["structure"]["structural_pass"] for row in serializable_paths),
    "active_returned_cache_gate_passed": all(
        row["structure"]["active_decode_cache_updates"] == NUM_LAYERS
        and row["structure"]["active_attention_boundary_checks"] == NUM_LAYERS
        for row in serializable_paths
    ),
    "eviction_first_gate_passed": (
        serializable_paths[3]["prefill_update_inputs"] == PYRAMID_BUDGETS
    ),
    "tq_ref_gate_passed": all(
        row["structure"]["tq_actual_path_max_nmse"] <= TQ_NMSE_MAX
        for row in serializable_paths if row["variant"] in {"Q3-Ref", "Q4-Ref"}
    ),
    "alias_deduplicated_storage_gate_passed": all(
        row["storage"]["probe_final_alias_deduplicated"] for row in serializable_paths
    ),
    "prefill_logit_agreement": prefill_agreement,
    "path_probes": serializable_paths,
    "tq_numerical_probe": tq_numerical_probe,
    "energy_rows": 0,
    "quality_rows": 0,
    "next_gate": (
        "Review these executed pilot artifacts. Only then may a separate full Qwen matrix notebook be built."
    ),
}
environment["status"] = "PILOT_EXECUTED_GATES_PASSED_VALIDATION_COMMIT_PENDING"
environment["validation_file"] = VALIDATION_JSON.name
atomic_write_json(ENVIRONMENT_JSON, environment)
validation["supporting_artifact_sha256"] = {
    PARITY_JSON.name: hashlib.sha256(PARITY_JSON.read_bytes()).hexdigest(),
    ENVIRONMENT_JSON.name: hashlib.sha256(ENVIRONMENT_JSON.read_bytes()).hexdigest(),
    PROMPT_MANIFEST_JSON.name: hashlib.sha256(PROMPT_MANIFEST_JSON.read_bytes()).hexdigest(),
}
atomic_write_json(VALIDATION_JSON, validation)  # Final atomic success marker.
release_model()
assert Qwen2Attention.forward is ORIGINAL_QWEN_FORWARD
print("All Qwen adapter/parity pilot gates passed:", EXPERIMENT_FINGERPRINT)


## 8. Completion Boundary

A successful final cell means only that the Qwen adapter and reference cache paths passed on this T4. It does not create scientific energy or quality evidence. Preserve the pilot directory for review, export the executed notebook, and keep `CSE407_qwen_track_a_reference_v1/` untouched. Build the separate full Qwen matrix notebook only after the validation JSON and prompt manifest have been reviewed and registered.
